# System Recommenders - Final Project 2025

### 🎯 Objective

Develop a recommender system that suggests short videos to users based on user preferences, interaction histories, and video content using the KuaiRec dataset. The challenge is to create a personalised and scalable recommendation engine similar to those used in platforms like TikTok or Kuaishou.

### 📥 Imports

In [101]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix
from implicit.als import AlternatingLeastSquares

plt.rcParams["figure.figsize"] = (20, 13)
%matplotlib inline
%config InlineBackend.figure_format = "retina"

### 📊 Download Dataset

We will use the **KuaiRec dataset**, a large-scale, fully-observed dataset collected from the Kuaishou short-video platform.

It contains:

- **User interactions** (views, likes, etc.)
- **Video metadata** (video ID, tags, etc.)
- **Timestamps**

More info: [KuaiRec Paper](https://arxiv.org/abs/2202.10842)

**Download dataset**

1. <ins>First option : Downloading Dataset via wget<ins>

In [102]:
%%bash
if [ ! -d "./data_final_project" ]; then
  wget --no-check-certificate 'https://drive.usercontent.google.com/download?id=1qe5hOSBxzIuxBb1G_Ih5X-O65QElollE&export=download&confirm=t&uuid=b2002093-cc6e-4bd5-be47-9603f0b33470' -O KuaiRec.zip
  unzip KuaiRec.zip -d ../data_final_project
else
  echo "Directory './data_final_project' already exists. Skipping download."
fi

Directory './data_final_project' already exists. Skipping download.


2. <ins>Second option : Downloading dataset via Google Drive<ins>

If the data is not downloaded by the wget because of a Connection Refused you might download it via this [link](https://drive.google.com/file/d/1qe5hOSBxzIuxBb1G_Ih5X-O65QElollE/view) and place it at the project root.

In [103]:
""" Uncomment if you need to (and if the first option did not work correctly)
%%bash
unzip KuaiRec.zip -d data_final_project
mv "data_final_project/KuaiRec 2.0/" data_final_project/KuaiRec
"""

' Uncomment if you need to (and if the first option did not work correctly)\n%%bash\nunzip KuaiRec.zip -d data_final_project\nmv "data_final_project/KuaiRec 2.0/" data_final_project/KuaiRec\n'

From this dataset we obtain the following files :

```bash
KuaiRec
  ├── data
  │   ├── big_matrix.csv          
  │   ├── small_matrix.csv
  │   ├── social_network.csv
  │   ├── user_features.csv
  │   ├── item_daily_features.csv
  │   └── item_categories.csv
  │   └── kuairec_caption_category.csv
```

- `interactions_train.csv`: historical user-item interactions for training.
- `interactions_test.csv`: user-item pairs to score during testing.
- `sample_submission.csv`: a template showing the expected output format.
- `video_metadata.csv`: metadata including tags or content-related features.

![image](img/KuaiRec.png)

## **1️⃣ Dataset Preprocessing**
📝 Associated Tasks :
- Load and inspect the dataset.
- Handle missing or inconsistent data.
- Merge metadata for content-based models if necessary.


💡 For more details about the different analyses and pre-processing decisions made on the available datasets, [go to the EDA notebook](./EDA/EDA.ipynb).

However, here is a small recap of the observations made : 

- **User Interactions**: 
  - The majority of users have **fewer than 3,000 interactions**.
  - There are a few **outliers** with interactions exceeding 6,000.

- **Item Popularity**:
  - There are very few items that receive **high levels of interaction**.
  - Many items, while not extremely popular, still receive a fair amount of attention.
  - A small subset of items are truly **highly popular**.

- **Time-Based Trends**:
  - However, interactions peak towards the **end of the week**, specifically from **Friday to Sunday**.
  - **Late-night hours (0:00 - 3:00 a.m.)** see the highest levels of activity.
  - **From 10:00 to 20:00** we see less activity
  - The time of day or day of the week has **minimal to no impact** on the watch ratio.

- **User Behavior**:
  - The **top 10 most active users** show a distinct dip in activity around **12:00 p.m.** and **4:00 p.m.**.

- **Video Length**:
  - **Shorter videos** (up to 30 seconds) receive **more interactions** and have a **higher watch ratio**.
  - Conversely, **longer videos** tend to have **fewer interactions** and a lower watch ratio.
  - Videos that are **no longer than 30 seconds** appear to have the optimal watch ratio.

- **Watch Ratio**:
  - **Half of the videos have less then 75% watch ratio.**
  - Only **0.1% of the videos have more than 18** of watch ratio. These are extreme outliers, we can either drop them or try and normalize them.

- **Video Type:**

    - **"Ad" videos have significantly lower interaction** rates compared to Normal videos, which are more engaging for users.
    - Users tend to engage less with promotional or advertisement-type content.

- **User Preferences:**

    - There is a clear preference for **Short Imports videos**. These videos are viewed and interacted with more often compared to longer or different upload types.

- **Video Format:**

    - The **1280x720 resolution is the optimal format** for maximizing user interaction and watch ratio. Other formats either fall short or show no significant improvement.

- **Video Privacy Settings:**

    - As expected, **public videos receive more interactions** than private or only friends videos. This could be due to the higher volume of public content being uploaded, which in turn increases the overall interactions.

    - Private or only friends videos tend to have a smaller, more targeted audience, reducing their interaction rates.

- **Video Age**:
    - **Recent videos (lower video age in days) tend to have more interactions**



### Load Datasets

Here as we have shown in the [EDA notebook](./EDA/EDA.ipynb) with more details :

1. We have to load the dataset we will use.

    We will only be using three datasets :
    - `big_matrix.csv`
        - Corresponds to the different interactions made on the videos,
        - It will serve for the training
    - `small_matrix.csv`
        - Same as big_matrix
        - it will serve for the testing
    - `item_daily_features`
        - Contains informations about the video (e.g. number of likes, shares, reports ...) 
        - It will be merged both with the small and big matrix to give additional informations

2. We make small preprocessing :
    - Converting some columns to a more appropriate type
    - Removing highly correlated columns
    - We will cap videos with an above 2.34 watch ratio as above are only the top 5% as show in the [EDA notebook](./EDA/EDA.ipynb). This will help normalize the `watch_ratio` which is an important metric.


In [104]:
interactions = pd.read_csv("./data_final_project/KuaiRec/data/big_matrix.csv")
small_interactions = pd.read_csv("./data_final_project/KuaiRec/data/small_matrix.csv")
item_features = pd.read_csv("./data_final_project/KuaiRec/data/item_daily_features.csv")

def clean_df(df):
    df = df.dropna()
    df = df.drop_duplicates()  
    return df  

def clean_df_timestamp(df):
    df = clean_df(df)
    df = df[df["timestamp"] >= 0]
    df = df[df["watch_ratio"] <= 200]
    return df

item_features = clean_df(item_features)
item_features = item_features.drop_duplicates(subset='video_id')
train_df = clean_df_timestamp(interactions)
test_df = clean_df_timestamp(small_interactions)

item_features['upload_dt'] = pd.to_datetime(item_features['upload_dt'])
item_features['date'] = pd.to_datetime(item_features['date'], format='%Y%m%d')


In [105]:
to_drop = ['date', 'play_duration', 'video_duration', 'time', 'timestamp']

train_df.drop(columns=to_drop, inplace=True, errors='ignore')
test_df.drop(columns=to_drop, inplace=True, errors='ignore')

In [106]:
train_df['watch_ratio'] = train_df['watch_ratio'].apply(lambda x: min(x, 2.34))
test_df['watch_ratio'] = test_df['watch_ratio'].apply(lambda x: min(x, 2.34))

In [107]:
correlation = item_features[[
       'video_duration', 'video_width',
       'video_height', 'music_id',
       'show_cnt', 'show_user_num', 'play_cnt', 'play_user_num',
       'play_duration', 'complete_play_cnt', 'complete_play_user_num',
       'valid_play_cnt', 'valid_play_user_num', 'long_time_play_cnt',
       'long_time_play_user_num', 'short_time_play_cnt',
       'short_time_play_user_num', 'play_progress', 'comment_stay_duration',
       'like_cnt', 'like_user_num', 'click_like_cnt', 'double_click_cnt',
       'cancel_like_cnt', 'cancel_like_user_num', 'comment_cnt',
       'comment_user_num', 'direct_comment_cnt', 'reply_comment_cnt',
       'delete_comment_cnt', 'delete_comment_user_num', 'comment_like_cnt',
       'comment_like_user_num', 'follow_cnt', 'follow_user_num',
       'cancel_follow_cnt', 'cancel_follow_user_num', 'share_cnt',
       'share_user_num', 'download_cnt', 'download_user_num', 'report_cnt',
       'report_user_num', 'reduce_similar_cnt', 'reduce_similar_user_num',
       'collect_cnt', 'collect_user_num', 'cancel_collect_cnt',
       'cancel_collect_user_num']].corr()

upper = correlation.where(np.triu(np.ones(correlation.shape), k=1).astype(bool))

to_drop = [column for column in upper.columns if any(upper[column] > 0.8)]
# These columns are dropped as we don't need them anymore 
# (merged in previous cell or just not needed for collaborative-filtering)
item_features.drop(columns=to_drop, inplace=True, errors='ignore')


## **2️⃣ Feature Engineering**
📝 Associated Tasks :
- Create meaningful features from interaction and metadata (e.g., content tags, user activity history).
- Build user-item interaction matrix.
- Optionally extract time-based or popularity-based features.



In [108]:
item_features['video_age'] = (item_features['date'] - item_features['upload_dt']).dt.days
item_features['is_short_video'] = (item_features['video_duration'].fillna(0) <= 30).astype(int)

We remove those columns as we don't need them in the future.

In [109]:
to_drop =   [
                'date', 'upload_dt', 'video_duration', 'music_id',
                'video_tag_name', 'play_progress', 'video_tag_id',
                'time', 'play_duration'
            ]
item_features.drop(columns=to_drop, inplace=True, errors='ignore')

In [110]:
train_df = pd.merge(train_df, item_features, on='video_id', how='left')
test_df = pd.merge(test_df, item_features, on='video_id', how='left')

As the ALS need some form of rating, we will use what we will call an engagement score. This is what the model will try to predict.

Based on our observations in the [EDA notebook](./EDA/EDA.ipynb), we can add some features such as :
- watch_ratio 
- video_type => if it is an "AD" then user are less likely to watch the video
- video_height and video_width => `1280x720` is the preferred format
- video_type => `ShortImports` are the preferred format
- visible_status => `public` video are more likely to be seen

In [111]:
def build_engagement_score(df):
    df["engagement_score"] = 0
    
    # Watch Ratio
    if 'watch_ratio' in df.columns:
        df["engagement_score"] += df['watch_ratio'].fillna(0) * 10
    
    # is_short_video
    df["engagement_score"] += df['is_short_video'].fillna(0) * 3
    
    # Video age
    if 'video_age' in df.columns:
        max_age = 365
        normalized_age = np.minimum(df['video_age'].fillna(max_age), max_age) / max_age
        # Newer videos get up to 2 points bonus
        df["engagement_score"] += (1 - normalized_age) * 2
    
    
    if 'video_type' in df.columns:
        df["engagement_score"] += np.where(
            df['video_type'] == 'AD',
            -3,  # penalty for ads
            2    # bonus for regular content
        )
    
    if 'visible_status' in df.columns:
        df["engagement_score"] += np.where(
            df['visible_status'] == 'public',
            2,  
            -1
        )
    
    if 'upload_type' in df.columns:
        upload_type_weights = {
            'ShortImport': 3,     # Short imported videos tend to be high quality
            'StartCamera': 2.5,   # Original camera content
            'Knowle': 2,          # Knowledge content
            'Web': 1.5,           # Web content
            'LongImport': 1,      # Long imported videos
            'UNKNOWN': 0,
            'LongCamera': 0.5,
            'PictureSet': 0.5,
            'LongPicture': 0.5,
            'ACurlVideo': 0.5,
            'followShot': 0.5,
            'ShareFromOtherApp': 0.5,
            'SameFrame': 0,
            'PictureCopy': 0,
            'FlashPhoto': 0,
            'PhotoCopy': 0,
            'LocalCollection': 0,
            'LocalInteraction': 0
        }
        df["engagement_score"] += df['upload_type'].map(upload_type_weights).fillna(0)
    
    engagement_columns = {
        'like_cnt': 0.5,
        'comment_cnt': 0.7,
        'share_cnt': 0.8,
        'collect_cnt': 0.6,
        'follow_cnt': 0.9,
        'complete_play_cnt': 0.7,
        'valid_play_cnt': 0.5,
        'reply_comment_cnt': 0.6,
        'comment_like_cnt': 0.4
    }
    
    for col, weight in engagement_columns.items():
        if col in df.columns:
            df["engagement_score"] += np.minimum(np.log1p(df[col].fillna(0)) * weight, 10)


    penalty_columns = {
        'cancel_like_cnt': 0.4,
        'cancel_follow_cnt': 0.5,
        'report_cnt': 0.7
    }
    for col, weight in penalty_columns.items():
        if col in df.columns:
            df["engagement_score"] -= np.minimum(np.log1p(df[col].fillna(0)) * weight, 10)

    if 'video_width' in df.columns:
        df["engagement_score"] += np.where(df['video_width'] >= 720, 0.5, 0)

    if 'video_height' in df.columns:
        df["engagement_score"] += np.where(df['video_height'] >= 1280, 0.5, 0)
    
    # Here we normalize the score    
    min_score = df["engagement_score"].min()
    max_score = df["engagement_score"].max()

    df["engagement_score"] = (df["engagement_score"] - min_score) / (max_score - min_score)
    
    return df

test_df = build_engagement_score(test_df)
train_df = build_engagement_score(train_df)

## **3️⃣ Model Development**
📝 Associated Tasks :
- Choose a recommendation approach:
    - Collaborative filtering (e.g., ALS, Matrix Factorisation)
    - Content-based filtering
    - Sequence-aware models
    - Hybrid approaches
- Train and validate your model on the training set.

As we have made an additional column called `engagement_score` this will act as the rating of our video. 

Therefore we can use ALS, which is a collaborative filtering model, that will try to guess the engagement score. The higher it is, the more engaging the video is.

The ALS Model was choosen here as it will take the different studied metrics in the one column `engagement_score` which will make more impact.

### User item matrix 
First we will create the user item matrix, which is essential for the ALS algorithm, to map the different users, videos and their engagement score.

In [112]:
# Get Unique user and videos
user_ids_train = train_df['user_id'].unique()
video_ids_train = train_df['video_id'].unique()

# Compute index for each user and videos
user_to_index = {user_id: idx for idx, user_id in enumerate(user_ids_train)}
video_to_index = {video_id: idx for idx, video_id in enumerate(video_ids_train)}

# add the index to the train and test
train_df['user_index'] = train_df['user_id'].map(user_to_index)
train_df['video_index'] = train_df['video_id'].map(video_to_index)

test_df['user_index'] = test_df['user_id'].map(user_to_index)
test_df['video_index'] = test_df['video_id'].map(video_to_index)

row = train_df['user_index'].values
col = train_df['video_index'].values

data = train_df['engagement_score'].values

n_users = train_df['user_index'].max() + 1
n_items = train_df['video_index'].max() + 1
    
user_item_matrix = csr_matrix((data, (row, col)), shape=(n_users, n_items))

### Model Training
Here we train our ALS Model with the `user_item_matrix` 💪

In [113]:
from implicit.als import AlternatingLeastSquares

model = AlternatingLeastSquares(
    factors=15,
    regularization=0.2,
    iterations=15,
    use_gpu=False,
    alpha=10
)

model.fit(user_item_matrix.T) 

/opt/homebrew/Caskroom/miniconda/base/lib/python3.12/site-packages/implicit/utils.py:164: ParameterWarning: Method expects CSR input, and was passed csc_matrix instead. Converting to CSR took 0.05760669708251953 seconds
  warnings.warn(


  0%|          | 0/15 [00:00<?, ?it/s]

## **4️⃣ Recommendation Algorithm**
📝 Associated Tasks :
- Predict which videos are likely to be enjoyed by each user in the test set.
- Generate a top-N ranked list of recommendations for each user.

In [114]:
top_n=100
def get_top_n_recommendations(model, user_item_matrix, user_ids, n=10, seen=True):
    recommendations = {}
    
    for user_id in user_ids:
        # Items the user has already interacted with (adjusted for training)
        already_interacted = set(user_item_matrix[user_id].indices) if seen else set()
        
        # U * V^T
        scores = model.user_factors[user_id].dot(model.item_factors.T)
       
        item_scores = [(item_id, scores[item_id])
                       for item_id in range(len(scores))
                       if item_id not in already_interacted]
        # Sort and select top-N items
        item_scores.sort(key=lambda x: x[1], reverse=True)
        top_items = [item[0] for item in item_scores[:n]]
        
        recommendations[user_id] = top_items
    print(item_scores[::-1])
    return recommendations

train_users = train_df['user_index'].unique()
test_users = test_df['user_index'].unique()

# For training eval, exclude seen items to simulate a true recommendation
train_recommendations = get_top_n_recommendations(
    model, user_item_matrix, train_users, n=top_n, seen=False  # Exclude seen items in training
)

# For test eval, include only unseen items (as per real-world recommendation)
test_recommendations = get_top_n_recommendations(
    model, user_item_matrix, test_users, n=top_n, seen=True  # Exclude seen items in test (real-world)
)

print(test_recommendations)

[(4901, -0.2979239), (5248, -0.2580265), (3924, -0.2517718), (388, -0.24406853), (4902, -0.24117549), (6534, -0.24060743), (6985, -0.23947708), (936, -0.2386954), (3412, -0.23565258), (2657, -0.23206821), (6273, -0.22703984), (6161, -0.22657856), (1872, -0.22378343), (6527, -0.2234297), (1176, -0.223104), (3507, -0.22171135), (4788, -0.21959083), (2981, -0.21828276), (6833, -0.21557584), (7059, -0.21527538), (273, -0.21241121), (6533, -0.21011874), (4142, -0.20956746), (2264, -0.20727928), (1565, -0.2060537), (2294, -0.2059734), (844, -0.20282498), (2922, -0.1987986), (2001, -0.19827391), (4483, -0.19735114), (644, -0.19441038), (2359, -0.19438063), (6295, -0.19381115), (6081, -0.18991382), (7149, -0.18813367), (2214, -0.18721384), (6556, -0.18654175), (2677, -0.18600565), (6127, -0.18544887), (4469, -0.18352327), (695, -0.18275951), (2077, -0.18262905), (208, -0.1822413), (5193, -0.18138142), (1982, -0.18121396), (5720, -0.18116273), (6815, -0.18101007), (5426, -0.1799321), (5301, -0.

## **5️⃣ Evaluation**
📝 Associated Tasks :
- Choose suitable metrics (e.g., Precision@K, Recall@K, MAP, NDCG).
- Evaluate performance and provide interpretations.

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix

def evaluate_recommendations_with_additional_metrics(recommendations, test_df, top_n=10, k=10):
    # Map of actual items per user
    user_actual_items = test_df.groupby('user_index')['video_index'].apply(set).to_dict()
    
    precision_at_n = []

    # Hit Rate, MRR, nDCG calculations
    hits = 0
    mrr = 0.0
    total_ndcg = 0.0
    count = 0
    
    for user_id, recommended_items in recommendations.items():
        if user_id in user_actual_items:
            actual_items = user_actual_items[user_id]
            recs_at_n = recommended_items[:top_n]

            # Precision
            num_relevant = len(set(recs_at_n) & actual_items)
            precision = num_relevant / len(recs_at_n) if recs_at_n else 0

            precision_at_n.append(precision)
            
            # Hit Rate
            gt = user_actual_items.get(user_id, set())
            if any(item in gt for item in recs_at_n[:k]):
                hits += 1

            # MRR
            for rank, item in enumerate(recs_at_n[:k], start=1):
                if item in gt:
                    mrr += 1.0 / rank
                    break

            # nDCG
            def dcg(recs, gt, k):
                return sum((1 / np.log2(i + 2)) if rec in gt else 0 for i, rec in enumerate(recs[:k]))

            def idcg(gt, k):
                n_relevant = min(len(gt), k)
                return sum(1 / np.log2(i + 2) for i in range(n_relevant))

            idcg_val = idcg(gt, k)
            if idcg_val > 0:
                total_ndcg += dcg(recs_at_n, gt, k) / idcg_val
                count += 1

    avg_precision = np.mean(precision_at_n) if precision_at_n else 0
    hit_rate = hits / len(user_actual_items) if user_actual_items else 0
    avg_mrr = mrr / len(user_actual_items) if user_actual_items else 0
    avg_ndcg = total_ndcg / count if count > 0 else 0

    return avg_precision, hit_rate, avg_mrr, avg_ndcg


# Create test user-item matrix for ground truth
test_row = test_df['user_index'].values
test_col = test_df['video_index'].values
test_data = np.ones(len(test_df))  # binary implicit feedback

test_user_item_matrix = csr_matrix((test_data, (test_row, test_col)), shape=(n_users, n_items))

def sparse_matrix_to_dict(matrix):
    user_item_dict = {}
    for user_id in range(matrix.shape[0]):
        items = matrix[user_id].indices
        if len(items) > 0:
            user_item_dict[user_id] = set(items)
    return user_item_dict

# Get ground truth from test matrix
ground_truth = sparse_matrix_to_dict(test_user_item_matrix)

# Evaluate on training and testing set
print("Evaluating on training set...")
train_precision, train_hit_rate, train_mrr, train_ndcg = evaluate_recommendations_with_additional_metrics(train_recommendations, train_df, top_n=top_n, k=10)
print(f"Training - Precision@{top_n}: {train_precision:.4f}, Hit Rate@{10}: {train_hit_rate:.4f}, MRR@{10}: {train_mrr:.4f}, nDCG@{10}: {train_ndcg:.4f}")

print("Evaluating on test set...")
test_precision, test_hit_rate, test_mrr, test_ndcg = evaluate_recommendations_with_additional_metrics(test_recommendations, test_df, top_n=top_n, k=10)
print(f"Testing - Precision@{top_n}: {test_precision:.4f}, Hit Rate@{10}: {test_hit_rate:.4f}, MRR@{10}: {test_mrr:.4f}, nDCG@{10}: {test_ndcg:.4f}")

Evaluating on training set...
Training - Precision@100: 0.1922, Recall@100: 0.0134, Hit Rate@10: 0.7885, MRR@10: 0.3599, nDCG@10: 0.1928
Evaluating on test set...
Testing - Precision@100: 0.4438, Recall@100: 0.0139, Hit Rate@10: 0.9972, MRR@10: 0.6292, nDCG@10: 0.4369


### Interpretation

We observe that precision over 100 recommendations is lower on the training set than on the testing set, which could be due to matrix sparsity.

The training set, composed of `big_matrix.csv`, is much larger than the testing set, which is based on `small_matrix.csv`. As a result, there are more videos to recommend in the training set, which might impact precision.

- **Precision@100**  
  19% of the videos recommended in the training set and around 44% in the test set are relevant in the top 100 recommendations.

- **Hit Rate@10**  
  79% of users in the training set and 99% in the test set received at least one relevant item in their top 10 recommendations.

- **MRR@10**  
  The mean reciprocal rank for the top 10 recommendations is **0.3612** on the training set and **0.6189** on the test set.  
  On average, a relevant item appears around the **3rd or 4th** position in the top 10 for the training set, and around the **6th or 7th** position for the test set.

- **nDCG@10**  
  The normalized discounted cumulative gain at rank 10 is **0.1947** in the training set and **0.4360** in the test set.  
  This indicates that the top 10 recommendations are poorly ordered by relevance in the training set, but better ordered in the test set.

---

### Conclusion

- **Precision** is higher for this model on unseen data, suggesting good generalization.
- **Hit Rate** and **MRR** indicate the model consistently presents at least one relevant item in the top recommendations.
- **nDCG** shows that the model has **poor to moderate ranking quality**, especially in the training set.
